# Causal LM & Instruction Tuning

> GPT-style language modeling and instruction tuning with PEFT (LoRA)

This notebook demonstrates causal language model fine-tuning and instruction tuning with LoRA.

In [ ]:
#| default_exp text.causal_lm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from blurr.text.causal_lm import CausalLMLearner

## Standard Causal LM Fine-tuning

Fine-tune a GPT-2 model on custom text data:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'text': [
        'The quick brown fox jumps over the lazy dog.',
        'Once upon a time in a faraway land, there lived a princess.',
        'The weather today is sunny and warm with a gentle breeze.',
        'In the beginning, there was nothing but darkness and silence.',
        'The scientist made a groundbreaking discovery in her laboratory.',
        'A long time ago, in a galaxy far far away.',
    ],
})

learn = CausalLMLearner.from_df(
    df,
    'sshleifer/tiny-gpt2',  # tiny model for demo
    text_col='text',
    max_length=64,
    bs=2,
)

In [ ]:
learn.fit_one_cycle(1, 2e-4)

### Text Generation

In [ ]:
results = learn.blurr_generate(
    'Once upon a time',
    max_new_tokens=50,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)
print(results[0]['completion'])

## Instruction Tuning

Fine-tune a model to follow instructions. blurr automatically:
- Formats instruction/response pairs using a template
- Masks the prompt tokens so loss is computed only on responses

### From Instruction Data

In [ ]:
instruct_df = pd.DataFrame({
    'instruction': [
        'Summarize the following text.',
        'Translate to French.',
        'What is the capital of France?',
        'Write a haiku about nature.',
    ],
    'response': [
        'The text discusses recent advances in AI technology.',
        'Bonjour, comment allez-vous?',
        'The capital of France is Paris.',
        'Cherry blossoms fall / Gentle breeze through bamboo grove / Natures song endures',
    ],
})

learn = CausalLMLearner.from_df(
    instruct_df,
    'sshleifer/tiny-gpt2',
    instruction_col='instruction',
    response_col='response',
    mask_prompt=True,  # Only compute loss on responses
    max_length=128,
    bs=2,
)
learn.fit_one_cycle(1, 2e-4)

### Custom Prompt Template

In [ ]:
# Use a custom template
learn = CausalLMLearner.from_df(
    instruct_df,
    'sshleifer/tiny-gpt2',
    instruction_col='instruction',
    response_col='response',
    prompt_template='<|user|>\n{instruction}\n<|assistant|>\n{response}',
    max_length=128,
    bs=2,
)

## PEFT / LoRA Fine-tuning

For large models, use parameter-efficient fine-tuning with LoRA:

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,                      # LoRA rank
    lora_alpha=16,            # LoRA alpha
    target_modules=['c_attn'],  # GPT-2 attention layer
    lora_dropout=0.1,
)

learn = CausalLMLearner.from_df(
    df,
    'sshleifer/tiny-gpt2',
    text_col='text',
    peft_config=peft_config,
    max_length=64,
    bs=2,
)
learn.fit_one_cycle(1, 2e-4)

### Save & Load PEFT Adapters

In [ ]:
# Save adapter weights
# learn.save_peft('./my_lora_adapter')

# Load adapter on base model
# model = CausalLMLearner.load_peft('sshleifer/tiny-gpt2', './my_lora_adapter')

## Generation Parameters

```python
learn.blurr_generate(
    text,
    max_new_tokens=128,     # Max new tokens
    temperature=0.7,        # Lower = more focused
    top_p=0.9,             # Nucleus sampling
    top_k=50,              # Top-k sampling
    do_sample=True,        # Sampling vs greedy
    num_beams=1,           # Beam search width
    repetition_penalty=1.0, # Penalize repetition
)
```

## Supported Models

- **GPT-2** family
- **LLaMA** / **LLaMA 2** / **LLaMA 3**
- **Mistral** / **Mixtral**
- **Phi** / **Phi-2** / **Phi-3**
- Any `AutoModelForCausalLM` model